# Bronze — geração e ingestão sintética

Parte do bundle Financial Services Medallion. Dados sintéticos.
DATASET_ID e CATALOG_NAME devem ser iguais em todas as tarefas.
Cada tarefa lê suas entradas das tabelas Delta persistidas, sem depender de variáveis de outra sessão.
Destinos existentes provocam falha para evitar sobrescrita; use novo DATASET_ID em uma nova carga.
Rótulos aleatórios: métricas de ML demonstrativas.


In [ ]:
import json, math, random, re
from datetime import datetime, timedelta
from decimal import Decimal
from uuid import uuid4
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DecimalType, IntegerType

if 'spark' not in globals() or 'oidlUtils' not in globals():
    raise RuntimeError('Execute no runtime AIDP com compute anexado.')

def parameter(name, default):
    value = oidlUtils.parameters.getParameter(name, str(default))
    return str(default) if value is None else str(value).strip()

def ident(s):
    if not re.fullmatch(r'[A-Za-z_][A-Za-z0-9_]*', s):
        raise ValueError(f'Identificador inválido: {s!r}')
    return f'`{s}`'

catalog = parameter('CATALOG_NAME', 'default')
accounts = int(parameter('ACCOUNT_COUNT', 5000))
seed = int(parameter('SEED', 42))
num_trees = int(parameter('RF_NUM_TREES', 100))
max_depth = int(parameter('RF_MAX_DEPTH', 10))
liquid = parameter('ENABLE_LIQUID_CLUSTERING', 'false').lower() == 'true'
if not 100 <= accounts <= 5000 or not 1 <= num_trees <= 100 or not 1 <= max_depth <= 10:
    raise ValueError('Limites do demo: 100–5000 contas, 1–100 árvores, profundidade 1–10.')
run_id = parameter('DATASET_ID', 'bundle_dev')
if not re.fullmatch(r'[A-Za-z][A-Za-z0-9_]{0,47}', run_id):
    raise ValueError('DATASET_ID deve começar com letra e ter até 48 letras, números ou _.')
schemas = {layer: f'finance_{layer}_{run_id}' for layer in ('bronze', 'silver', 'gold')}
def table(layer, name):
    return '.'.join(ident(v) for v in (catalog, schemas[layer], name))
bronze_table = table('bronze', 'account_transactions_bronze')
silver_table = table('silver', 'account_transactions_silver')
analytics_table = table('gold', 'transaction_analytics_gold')
features_table = table('gold', 'fraud_detection_model_gold')
result = {'run_id': run_id, 'catalog': catalog, 'schemas': schemas, 'accounts': accounts, 'seed': seed, 'liquid_clustering_enabled': liquid}
def write_delta(frame, target, keys):
    if liquid:
        view = 'finance_stage_' + uuid4().hex[:8]
        frame.createOrReplaceTempView(view)
        spark.sql(f'CREATE TABLE {target} USING DELTA CLUSTER BY ({keys}) AS SELECT * FROM {view}')
        spark.catalog.dropTempView(view)
    else:
        frame.write.format('delta').mode('errorifexists').saveAsTable(target)

def ensure_schema(layer):
    spark.sql(f'CREATE SCHEMA IF NOT EXISTS {ident(catalog)}.{ident(schemas[layer])}')

def require_table(target):
    if not spark.catalog.tableExists(target):
        raise RuntimeError(f'Execute a camada anterior primeiro: {target}')


In [ ]:
ensure_schema('bronze')
# Geração determinística de transações e schema explícito, inclusive Decimal e timestamp.
rng = random.Random(seed)
transaction_types = ['Deposit', 'Withdrawal', 'Transfer', 'Payment', 'ATM']
merchants = ['Retail', 'Restaurant', 'Online', 'Utilities', 'Entertainment', 'Groceries', 'Healthcare', 'Transportation']
locations = ['New York, NY', 'Los Angeles, CA', 'Chicago, IL', 'Houston, TX', 'Miami, FL', 'Online', 'ATM']
base_date = datetime(2024, 1, 1)
ingestion_time = datetime.utcnow()
records = []
for account_num in range(1, accounts + 1):
    for _ in range(rng.randint(10, 50)):
        date = base_date + timedelta(days=rng.randint(0, 365), hours=rng.randint(0, 23))
        kind = rng.choice(transaction_types)
        low, high, sign = (100, 10000, 1) if kind in ('Deposit', 'Transfer') else ((20, 500, -1) if kind == 'ATM' else (10, 2000, -1))
        amount = Decimal(str(round(rng.uniform(low, high) * sign, 2))).quantize(Decimal('0.01'))
        merchant = rng.choice(merchants)
        location = 'ATM' if kind == 'ATM' else ('Online' if merchant == 'Online' else rng.choice(locations))
        records.append((f'ACC{account_num:08d}', date, kind, amount, merchant, location, rng.randint(0, 100), ingestion_time))
raw_schema = StructType([
    StructField('account_id', StringType()), StructField('transaction_date', TimestampType()),
    StructField('transaction_type', StringType()), StructField('amount', DecimalType(15, 2)),
    StructField('merchant_category', StringType()), StructField('location', StringType()),
    StructField('risk_score', IntegerType()), StructField('ingestion_timestamp', TimestampType())
])
raw = spark.createDataFrame(records, raw_schema)
expected_rows = len(records)

write_delta(raw, bronze_table, 'account_id, transaction_date')
bronze = spark.table(bronze_table)
bronze_count = bronze.count()
assert bronze_count == expected_rows
result['bronze_rows'] = bronze_count
print(f'FINANCE_STAGE=bronze_complete rows={bronze_count}')

In [ ]:

result.update(stage='bronze', status='SUCCESS', synthetic_labels_only=True)
result['tables'] = [t.replace('`', '') for t in (bronze_table,)]
print('AIDP_FINANCE_BUNDLE_STAGE_RESULT=' + json.dumps(result, sort_keys=True))
